[![Open In Colab](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/badge/open-in-colab.svg)](https://colab.research.google.com/github/crunchdao/quickstarters/blob/master/competitions/structural-break-real-time/quickstarters/dummy/dummy.ipynb)

![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/master/competitions/structural-break-real-time/assets/banner.webp)

# ADIA Lab Structural Break Streaming Data Challenge

Lorem ipsum dolor sit amet, consectetur adipiscing elit.

# Setup

The first steps to get started are:
1. Get the setup command
2. Execute it in the cell below

### >> https://hub.crunchdao.io/competitions/structural-break-real-time/submit/notebook

![Reveal token](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/reveal-token.gif)

In [ ]:
# Install the Crunch CLI
%pip install --upgrade crunch-cli

# Setup your local environment
!crunch setup --notebook structural-break-real-time hello --token aaaabbbbccccddddeeeeffff

In [ ]:
# Necessary for the staging environment
#%env CRUNCH_COMPETITIONS_DIRECTORY_PATH=../../../../
%env CRUNCH_COMPETITIONS_BRANCH=feat/structural-break-2
%env API_BASE_URL=https://api.hub.crunchdao.io/
%env WEB_BASE_URL=https://hub.crunchdao.io/

import crunch.store
crunch.store.load_from_env()

# Your model

## Setup

In [17]:
import os
from statistics import mean
from typing import Iterable, List, Optional, Tuple

# Import your dependencies
import joblib
import pandas as pd
import scipy
from sklearn.metrics import roc_auc_score

In [ ]:
import crunch

# Load the Crunch Toolings
crunch_tools = crunch.load_notebook()

## Understanding the Data

Lorem ipsum dolor sit amet, consectetur adipiscing elit.

In [ ]:
# Load the data simply
train_data, test_data = crunch_tools.load_data()

### Understanding `train_data`

The training data is structured as a sequence of tuples containing:
- `dataset_id`: Identifies the unique time series;
- `x_historical`: Values of the **historical** segment.
- `x_online`: Values of the **online** segment.
- `tau_index`: Index at which the structural break happened in `x_online`. Or `None` if the time series does not have one.

<br />

It has been designed this way so that you can easily iterate over it:
```python
for dataset_id, x_historical, x_online, tau_index in datasets:
    # dataset_id: int
    # x_historical: numpy.ndarray[float]
    # x_online: numpy.ndarray[float]
    # tau_index: Optional[int]

    ...
```

<br />

> **👉 Tip** <br />
> ---
> Avoid trying to print the whole `train_data` value. <br />
> Doing so might cause your IDE to crash.

In [9]:
(
    dataset_id,
    x_historical,
    x_online,
    tau_index,
) = train_data[6]

print("dataset_id", dataset_id)
print("x_historical")
print("  -> len:", len(x_historical))
print("  -> sample:", repr(x_historical[:3]))
print("x_online")
print("  -> len:", len(x_online))
print("  -> sample:", repr(x_online[:3]))
print("tau_index", tau_index)

dataset_id 6
x_historical
  -> len: 7452
  -> sample: array([0.02505864, 2.02345746, 3.6296637 ])
x_online
  -> len: 986
  -> sample: array([-1.65793553, -1.39104332, -0.93936132])
tau_index 256


### Understanding `test_data`

The training data is structured as a sequence of tuples containing:
- `x_historical`: Values of the **historical** segment.
- `x_online`: Values of the **online** segment.

<br />

> **⚠ Warning** <br />
> ---
> You may read the datasets and `x_online` values multiple times during local testing. <br />
> **However, in the cloud environment, you will only be able to read them once and one at a time.**

In [12]:
(
    x_historical,
    x_online,
) = test_data[0]

print("x_historical")
print("  -> len:", len(x_historical))
print("  -> sample:", repr(x_historical[:3]))
print("x_online")
print("  -> len:", len(x_online))
print("  -> sample:", repr(x_online[:3]))

x_historical
  -> len: 6587
  -> sample: array([ 0.06361621, -0.56514349, -0.7648987 ])
x_online
  -> len: 317
  -> sample: array([1.12896977, 0.35890231, 0.12529747])


## Strategy Implementation

Lorem ipsum dolor sit amet, consectetur adipiscing elit.

### The `train()` Function

In this function, you build and train your model for making inferences on the test data. Your model must be stored in the `model_directory_path`.

The baseline implementation below doesn't require a pre-trained model, as it uses a statistical test that will be computed at inference time.

In [25]:
def train(
    datasets: List[Tuple[int, List[float], List[float], Optional[int]]],
    model_directory_path: str,
):
    model = None

    for dataset_id, x_historical, x_online, tau_index in datasets:
        pass

    joblib.dump(model, os.path.join(model_directory_path, 'model.joblib'))

### The `infer()` Function

In the inference function, your trained model (if any) is loaded and used to make predictions on test data.

**Workflow:**
1. Load your model;
2. Use the `yield` statement to signal readiness to the runner;
3. Process each dataset one by one within the for loop;
4. Process each points of `x_online`;
5. For each value, use `yield prediction` to return your prediction.

<br />

Below is an empty example of what is expected:
```python
...  # Load your model (if any)

yield  # Mark as ready

for x_historical, x_online in datasets:
    ...  # Prepare your state for the dataset

    for point in x_online:
        ...  # Consume the point

        yield 0.5  # Yield your prediction
```

<br />

> **📝 Note** <br />
> ---
> The `datasets` object AND each `x_online` can only be iterated once! <br />
> **You must `yield` at each loop of `x_online`.**

In [24]:
def infer(
    datasets: Iterable[Tuple[List[float], Iterable[float]]],
    model_directory_path: str,
):
    model = joblib.load(os.path.join(model_directory_path, 'model.joblib'))

    yield  # Mark as ready

    for x_historical, x_online in datasets:
        historical_mean = mean(x_historical)
        found = False

        online_sum = 0.0
        online_count = 0

        for point in x_online:

            # Avoid unnecessary calculations once a break is found
            if not found:

                # Accumulate mean incrementally
                online_sum += point
                online_count += 1
                online_mean = online_sum / online_count

                # Only make a decision once we have enough online points to be confident
                if online_count >= 10:

                    # Compare the online mean to the historical mean
                    ratio = online_mean / historical_mean

                    # If the online mean is significantly (20%) different from the historical mean, we predict a break
                    found = ratio > 1.2 or ratio < 0.8

            yield float(found)

## Local testing

To make sure your `train()` and `infer()` function are working properly, you can call the `crunch.test()` function that will reproduce the cloud environment locally. <br />
Even if it is not perfect, it should give you a quick idea if your model is working properly.

In [ ]:
crunch_tools.test(
    # Uncomment to disable the train
    # force_first_train=False,

    # Uncomment to disable the determinism check
    # no_determinism_check=True,
)

## Results

Once the local tester is done, you can preview the result stored in `prediction/prediction.parquet`.

In [27]:
prediction = pd.read_parquet("prediction/prediction.parquet")
prediction

prediction
id    time            
10000 6587         0.0
      6588         0.0
      6589         0.0
      6590         0.0
      6591         0.0
...                ...
10099 4786         1.0
      4787         1.0
      4788         1.0
      4789         1.0
      4790         1.0

[50983 rows x 1 columns]

### Local scoring

To evaluate your solution on the reduced test set, merge your prediction with the true labels and call `roc_auc_score` separately at each time step, then take the weighted average.

In [28]:
# Load the targets
y_test = pd.read_parquet("data/y_test.reduced.parquet")

# Merge predictions with true labels
merged = prediction.merge(
    y_test,
    how="left",
    left_index=True,
    right_index=True
)

# Add a column for the online time step (0, 1, 2, ...)
merged["time_online"] = merged.groupby("id").cumcount()

weighted_auc_sum = 0.0
total_weight = 0.0

# Step 1: select all observations at time t
for time, group in merged.groupby("time_online"):
    labels = group["target"].values
    scores = group["prediction"].values

    # Step 2: count positives and negatives
    n_positive = int(labels.sum())
    n_negative = int((1 - labels).sum())

    # Step 3: skip if only one class is present
    if n_positive == 0 or n_negative == 0:
        continue

    # Step 4: AUC at this time step and its weight
    auc_at_time = float(roc_auc_score(labels, scores))
    weight = float(n_positive * n_negative)

    weighted_auc_sum += weight * auc_at_time
    total_weight += weight

if total_weight == 0.0:
    score = 0.5
else:
    score = weighted_auc_sum / total_weight

print(f"TS-AUC Score: {score:.4f}")

TS-AUC Score: 0.5000


# Submit your Notebook

To submit your work, you must:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Create a run to validate it

Executing the cell below will take care of everything (only available on Google Colab), or show you how to submit manually.

In [ ]:
# @title  {"display-mode":"form", "form-width":"400px"}

# @markdown Describe your changes, then run the cell.
Message = "" # @param {"type":"string","placeholder":"Short description (optional)"}

# ---
# THIS METHOD IS ONLY POSSIBLE ON COLAB.
# RUNNING THIS CELL WILL PROMPT YOU TO USE THE OLD WAY OF SUBMITTING A NOTEBOOK.

crunch_tools.submit(
    message=Message,
)